# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [8]:
# Import necessary libraries
import math
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from deepforest import main
from shapely.geometry import Polygon

# Import your chosen baseline model
# Example: from sklearn.linear_model import LogisticRegression


## Model Choice

We use **DeepForest** as our baseline model, a pretrained deep learning model 
for tree crown detection based on RetinaNet, trained on the NEON dataset 
(North American forests, ~10 cm resolution).

The model is applied **zero-shot**, meaning no fine-tuning on the Denmark dataset. 
This is intentional: the baseline is designed to represent the performance achievable 
without any domain-specific training, establishing a lower bound that our fine-tuned 
model in Step 3 must exceed.

Limitations are expected: DeepForest was trained on RGB data from a different 
continent, different tree species, and a different resolution (10 cm vs. 20 cm). 
Weak performance on the Denmark test set is therefore anticipated and meaningful.


## Feature Selection

DeepForest accepts **RGB input only**. Of the six available channels 
(Red, Green, Blue, Infrared, NDVI, CHM), only the three visible bands are used.

This is a deliberate constraint of the baseline. Notably, CHM is the strongest 
single predictor of tree presence (r = 0.37, EDA) and is entirely excluded here. 
The gap between this RGB-only zero-shot baseline and our Step 3 model, 
which uses all six channels, directly quantifies the value of domain-specific 
training and richer input features.

In [9]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
DATA_ROOT = Path('../Data')
TEST_DIR = DATA_ROOT / 'extracted_data_2aux_test_v4_centroids_all_classes_final'


# CHANNELS = ['red', 'green', 'blue']

## Implementation

[Implement your baseline model here.]



In [21]:
model = main.deepforest()
model.load_model("weecology/deepforest-tree")
model.config["score_thresh"] = 0.1

all_predictions = {}
for red_path in sorted(TEST_DIR.glob('red_*.png')):
    idx = red_path.stem.split('_')[1]  # '0', '1', '2', ...
    ann_path = TEST_DIR / f'annotation_{idx}.json'
    
    red   = np.array(Image.open(TEST_DIR / f'red_{idx}.png'))
    green = np.array(Image.open(TEST_DIR / f'green_{idx}.png'))
    blue  = np.array(Image.open(TEST_DIR / f'blue_{idx}.png'))
    
    rgb = np.stack([red, green, blue], axis=-1)
    predictions = model.predict_image(image=rgb)
    all_predictions[idx] = predictions

pred_circles = {}
gt_circles = {}

for idx, predictions in all_predictions.items():
    gt = []
    for coords in ann.get('Trees', []):
        poly = Polygon(coords)
        cx = poly.centroid.x
        cy = poly.centroid.y
        r  = (poly.area / 3.14159) ** 0.5
        gt.append((cx, cy, r))
    gt_circles[idx] = gt
    
    if predictions is None:
        pred_circles[idx] = []
        continue
    
    circles = []
    for _, row in predictions.iterrows():
        cx = (row['xmin'] + row['xmax']) / 2
        cy = (row['ymin'] + row['ymax']) / 2
        r  = (row['xmax'] - row['xmin']) / 2
        circles.append((cx, cy, r))
    
    pred_circles[idx] = circles

    with open(TEST_DIR / f'annotation_{idx}.json') as f:
        ann = json.load(f)
    


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/tmp/ipykernel_5262/1604634978.py:17: UserWarning: An image was passed directly to predict_image, the result.root_dir attribute will be None in the output dataframe, to use visualize.plot_results, please assign results.root_dir = <directory name>
  predictions = model.predict_image(image=rgb)


0
10
11
13
14
15
16
18
19
20
21
23
24
5
6
8


## Evaluation

We evaluate using **Precision, Recall, F1, and mean IoU** over matched 
predicted-GT circle pairs.

Ground-truth polygons are converted to area-equivalent circles 
(centroid + radius = √(A/π)). DeepForest bounding boxes are converted to circles 
using the box centre as centroid and half the box width as radius.

Matching is greedy: each prediction is matched to its best available 
ground-truth circle. A match counts as a true positive if circle IoU ≥ 0.5.

This evaluation pipeline is model-agnostic and will be reused unchanged in Step 3.

In [22]:
# Evaluate the baseline model
# Example for a classification problem
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)

# For a regression problem, you might use:
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


def circle_iou(c1, c2):
    x1, y1, r1 = c1
    x2, y2, r2 = c2
    d = math.sqrt((x1-x2)**2 + (y1-y2)**2)
    
    if d >= r1 + r2:
        return 0.0
    if d <= abs(r1 - r2):
        return min(r1,r2)**2 / max(r1,r2)**2
    
    a = 2 * math.acos(min(1, (d**2 + r1**2 - r2**2) / (2*d*r1)))
    b = 2 * math.acos(min(1, (d**2 + r2**2 - r1**2) / (2*d*r2)))
    intersection = 0.5*r1**2*(a - math.sin(a)) + 0.5*r2**2*(b - math.sin(b))
    union = math.pi*r1**2 + math.pi*r2**2 - intersection
    return intersection / union


def evaluate(preds, gts, iou_threshold=0.5):
    if not gts:
        return dict(precision=0.0, recall=0.0, f1=0.0, tp=0, fp=len(preds), fn=0)
    if not preds:
        return dict(precision=0.0, recall=0.0, f1=0.0, tp=0, fp=0, fn=len(gts))
    
    matched = set()
    tp = 0
    for pred in preds:
        best_iou, best_j = 0, -1
        for j, gt in enumerate(gts):
            if j in matched:
                continue
            iou = circle_iou(pred, gt)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_threshold:
            matched.add(best_j)
            tp += 1
    
    fp = len(preds) - tp
    fn = len(gts) - tp
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1        = 2*precision*recall / (precision+recall) if precision+recall > 0 else 0.0
    return dict(precision=precision, recall=recall, f1=f1, tp=tp, fp=fp, fn=fn)


results = []
for idx in pred_circles:
    m = evaluate(pred_circles[idx], gt_circles[idx])
    m['idx'] = idx
    results.append(m)

df = pd.DataFrame(results)
print(f'Precision : {df["precision"].mean():.4f}')
print(f'Recall    : {df["recall"].mean():.4f}')
print(f'F1        : {df["f1"].mean():.4f}')
print(f'TP        : {df["tp"].sum()}')
print(f'FP        : {df["fp"].sum()}')
print(f'FN        : {df["fn"].sum()}')    


Precision : 0.0000
Recall    : 0.0000
F1        : 0.0000
TP        : 0
FP        : 18
FN        : 3858
